# 04 — Similarity Search Experiments

## Retrieval Configuration: Comparing `k`

**Experiment IDs:** SS-001, SS-002, SS-003, SS-004

### Objective

Evaluate how the number of retrieved documents (`k`) affects similarity search results using the embedding model selected in Notebook 03.

### Selected Embedding Baseline

```text
Embedding Model : text-embedding-3-large
Chunk Size      : 500
Chunk Overlap   : 50
Documents       : 300
Chunks          : 1,538
```

### What Are We Testing?

In Notebook 03 we compared embedding models.

Now we keep the embedding model fixed and change only:

```text
k = number of documents returned by similarity search
```

We will test:

```text
k = 1
k = 3
k = 5
k = 10
```

### Why This Experiment Matters

Choosing a very small `k` may return insufficient context.

Choosing a very large `k` may introduce:

- irrelevant chunks
- duplicate information
- unnecessary context
- higher downstream LLM processing cost

The objective is therefore not:

> "The largest k is always better."

Instead, we want to identify a practical retrieval configuration for our dataset.

## 1. Environment & Imports

This notebook uses the same OpenAI embedding model selected in Notebook 03.

No new model is introduced here.

Keeping the embedding model fixed makes this a controlled retrieval experiment.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY environment variable is not set. "
        "Please configure it in your .env file."
    )

print("Environment configured successfully.")

C:\Users\visha\AppData\Local\Temp\ipykernel_32096\2646475630.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


Environment configured successfully.


## 2. Load the Canonical Dataset

The same dataset used throughout the project is loaded explicitly.

Expected source documents:

```text
300
```

In [2]:
DATA_PATH = "../data/Pharma_Sales_Long.csv"

loader = CSVLoader(
    file_path=DATA_PATH,
    encoding="utf-8"
)

data = loader.load()

assert len(data) == 300, (
    f"Expected 300 documents, but found {len(data)}."
)

print(f"Loaded {len(data)} documents.")

Loaded 300 documents.


## 3. Recreate the Chunking Baseline

Notebook 02 selected:

```text
Chunk Size    : 500
Chunk Overlap : 50
Expected      : 1,538 chunks
```

This must remain unchanged for the retrieval experiment.

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(data)

assert len(chunks) == 1538, (
    f"Expected 1,538 chunks, but found {len(chunks)}."
)

print(f"Documents : {len(data)}")
print(f"Chunks    : {len(chunks)}")
print("✓ Chunking baseline validated.")

Documents : 300
Chunks    : 1538
✓ Chunking baseline validated.


## 4. Create the Selected Vector Store

Notebook 03 selected:

```text
text-embedding-3-large
```

We create one Chroma vector store using the 1,538 baseline chunks.

The vector store remains constant for all `k` experiments.

Only the retrieval parameter changes.

In [4]:
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="similarity_search_baseline",
)

print("✓ Chroma vector store created.")
print(f"✓ Stored chunks: {len(chunks)}")

✓ Chroma vector store created.
✓ Stored chunks: 1538


## 5. Define the Evaluation Questions

We intentionally reuse the same four questions from Notebook 03.

This is important because it allows us to compare:

```text
Embedding Experiment
        ↓
Same Questions
        ↓
Retrieval Experiment
```

We do not introduce new questions here.

In [5]:
evaluation_questions = {
    "Q001": (
        "Which sales records describe WELIREG discussions related to renal cell "
        "carcinoma in territories where customer engagement or follow-up activity "
        "was also mentioned?"
    ),
    "Q002": (
        "Find records where the sales discussion combines physician engagement, "
        "approved clinical information, and follow-up planning for a vaccine product."
    ),
    "Q003": (
        "Which records indicate regional or territory-level business opportunities "
        "while also referring to prescription trends and customer behavior?"
    ),
    "Q004": (
        "Find records where customer discussions include market access or competitor "
        "comparison together with compliant promotional or scientific-literature activities."
    ),
}

for question_id, question in evaluation_questions.items():
    print("=" * 80)
    print(question_id)
    print(question)

Q001
Which sales records describe WELIREG discussions related to renal cell carcinoma in territories where customer engagement or follow-up activity was also mentioned?
Q002
Find records where the sales discussion combines physician engagement, approved clinical information, and follow-up planning for a vaccine product.
Q003
Which records indicate regional or territory-level business opportunities while also referring to prescription trends and customer behavior?
Q004
Find records where customer discussions include market access or competitor comparison together with compliant promotional or scientific-literature activities.


## 6. Define `k` Configurations

We test four retrieval sizes.

| Experiment | k | Purpose |
|---|---:|---|
| SS-001 | 1 | Only the highest-ranked result |
| SS-002 | 3 | Small context window |
| SS-003 | 5 | Moderate context window |
| SS-004 | 10 | Larger context window |

### Expected Behavior

As `k` increases:

```text
Retrieved context ↑
Potential coverage ↑
Potential noise ↑
```

The experiment is therefore a trade-off analysis.

In [6]:
k_configurations = [
    {
        "experiment_id": "SS-001",
        "k": 1,
    },
    {
        "experiment_id": "SS-002",
        "k": 3,
    },
    {
        "experiment_id": "SS-003",
        "k": 5,
    },
    {
        "experiment_id": "SS-004",
        "k": 10,
    },
]

for config in k_configurations:
    print(
        f"{config['experiment_id']} | "
        f"k = {config['k']}"
    )

SS-001 | k = 1
SS-002 | k = 3
SS-003 | k = 5
SS-004 | k = 10


## 7. Run Similarity Search Experiments

For every question and every `k` value we record:

- Rank
- Relevance score
- Source row
- Content preview

The embedding model and vector store remain unchanged.

In [7]:
similarity_results = []

for config in k_configurations:

    experiment_id = config["experiment_id"]
    k = config["k"]

    print("=" * 80)
    print(f"Running {experiment_id} | k={k}")

    for question_id, question in evaluation_questions.items():

        results = vector_store.similarity_search_with_relevance_scores(
            question,
            k=k
        )

        for rank, (document, score) in enumerate(results, start=1):

            similarity_results.append({
                "Experiment_ID": experiment_id,
                "K": k,
                "Question_ID": question_id,
                "Rank": rank,
                "Score": float(score),
                "Source_Row": document.metadata.get("row"),
                "Content_Preview": document.page_content[:250],
            })

    print(f"Completed {experiment_id}.")

similarity_results_df = pd.DataFrame(similarity_results)

print("=" * 80)
print("All similarity search experiments completed.")

Running SS-001 | k=1
Completed SS-001.
Running SS-002 | k=3
Completed SS-002.
Running SS-003 | k=5
Completed SS-003.
Running SS-004 | k=10
Completed SS-004.
All similarity search experiments completed.


## 8. Inspect the Ranking for One Question

Before aggregating results, inspect the actual ranking.

Q001 is used here because it produced the strongest retrieval result in Notebook 03.

This allows us to see how additional results enter the retrieval set as `k` increases.

In [8]:
for experiment_id in [
    "SS-001",
    "SS-002",
    "SS-003",
    "SS-004",
]:

    print("=" * 80)
    print(f"{experiment_id}")

    results = similarity_results_df[
        (
            similarity_results_df["Experiment_ID"] == experiment_id
        )
        & (
            similarity_results_df["Question_ID"] == "Q001"
        )
    ].sort_values("Rank")

    for _, row in results.iterrows():
        print(
            f"Rank {row['Rank']:>2} | "
            f"Row {row['Source_Row']} | "
            f"Score {row['Score']:.4f}"
        )

SS-001
Rank  1 | Row 13 | Score 0.6911
SS-002
Rank  1 | Row 13 | Score 0.6911
Rank  2 | Row 135 | Score 0.6911
Rank  3 | Row 237 | Score 0.6906
SS-003
Rank  1 | Row 13 | Score 0.6911
Rank  2 | Row 135 | Score 0.6911
Rank  3 | Row 81 | Score 0.6906
Rank  4 | Row 237 | Score 0.6906
Rank  5 | Row 283 | Score 0.6906
SS-004
Rank  1 | Row 13 | Score 0.6911
Rank  2 | Row 135 | Score 0.6911
Rank  3 | Row 1 | Score 0.6906
Rank  4 | Row 71 | Score 0.6906
Rank  5 | Row 80 | Score 0.6906
Rank  6 | Row 81 | Score 0.6906
Rank  7 | Row 85 | Score 0.6906
Rank  8 | Row 132 | Score 0.6906
Rank  9 | Row 237 | Score 0.6906
Rank 10 | Row 283 | Score 0.6906


## 9. Compare Top-1 and Average Retrieved Scores

For each question and `k`, calculate:

- Top-1 score
- Average score of all retrieved documents

### Why Both?

**Top-1 score**

Tells us how strong the best result is.

**Average retrieved score**

Tells us how relevant the complete retrieved set is on average.

As `k` increases, the average score may decline because lower-ranked documents are included.

In [9]:
retrieval_summary = (
    similarity_results_df
    .groupby(
        ["Experiment_ID", "K", "Question_ID"],
        as_index=False
    )
    .agg(
        Top1_Score=("Score", "max"),
        Average_Retrieved_Score=("Score", "mean"),
        Lowest_Retrieved_Score=("Score", "min"),
    )
)

display(retrieval_summary)

,Experiment_ID,K,Question_ID,Top1_Score,Average_Retrieved_Score,Lowest_Retrieved_Score
0,SS-001,1,Q001,0.691113,0.691113,0.691113
1,SS-001,1,Q002,0.447207,0.447207,0.447207
2,SS-001,1,Q003,0.381181,0.381181,0.381181
3,SS-001,1,Q004,0.339957,0.339957,0.339957
4,SS-002,3,Q001,0.691113,0.690941,0.690597
5,SS-002,3,Q002,0.447207,0.447207,0.447207
6,SS-002,3,Q003,0.381181,0.381115,0.381081
7,SS-002,3,Q004,0.339957,0.339957,0.339957
8,SS-003,5,Q001,0.691113,0.690804,0.690597
9,SS-003,5,Q002,0.447207,0.447197,0.447175


## 10. Overall `k` Comparison

Now aggregate the results across all four questions.

This provides a high-level view of the trade-off between retrieval depth and average relevance.

In [10]:
k_summary = (
    retrieval_summary
    .groupby(
        ["Experiment_ID", "K"],
        as_index=False
    )
    .agg(
        Average_Top1_Score=("Top1_Score", "mean"),
        Average_Retrieved_Score=("Average_Retrieved_Score", "mean"),
        Average_Lowest_Score=("Lowest_Retrieved_Score", "mean"),
    )
    .sort_values("K")
)

display(k_summary)

,Experiment_ID,K,Average_Top1_Score,Average_Retrieved_Score,Average_Lowest_Score
0,SS-001,1,0.464864,0.464864,0.464864
1,SS-002,3,0.464864,0.464805,0.464711
2,SS-003,5,0.464864,0.464764,0.464702
3,SS-004,10,0.464864,0.464733,0.464702


## 11. Score Drop as `k` Increases

The Top-1 result does not change when `k` changes because the first-ranked result remains the first-ranked result.

The more useful metric here is the average relevance of the entire retrieved set.

We calculate the difference from the `k=1` baseline.

### Interpretation

If the average score drops significantly as `k` increases, the additional results are becoming progressively less relevant.

In [11]:
baseline_k1 = k_summary[
    k_summary["K"] == 1
]["Average_Retrieved_Score"].iloc[0]

k_summary["Difference_vs_K1"] = (
    k_summary["Average_Retrieved_Score"]
    - baseline_k1
)

k_summary["Difference_vs_K1_%"] = (
    k_summary["Difference_vs_K1"]
    / baseline_k1
    * 100
)

display(k_summary)

,Experiment_ID,K,Average_Top1_Score,Average_Retrieved_Score,Average_Lowest_Score,Difference_vs_K1,Difference_vs_K1_%
0,SS-001,1,0.464864,0.464864,0.464864,0.000000,0.000000
1,SS-002,3,0.464864,0.464805,0.464711,-0.000060,-0.012808
2,SS-003,5,0.464864,0.464764,0.464702,-0.000100,-0.021455
3,SS-004,10,0.464864,0.464733,0.464702,-0.000131,-0.028123


## 12. Question-Level `k` Comparison

Overall averages can hide important behavior.

We therefore compare each question separately.

This helps identify whether a particular question benefits from retrieving more context.

In [12]:
question_k_comparison = (
    retrieval_summary
    .pivot(
        index="Question_ID",
        columns="K",
        values="Average_Retrieved_Score"
    )
)

display(question_k_comparison)

K,1,3,5,10
Question_ID,,,,
Q001,0.691113,0.690941,0.690804,0.690700
Q002,0.447207,0.447207,0.447197,0.447186
Q003,0.381181,0.381115,0.381101,0.381091
Q004,0.339957,0.339957,0.339957,0.339957


## 13. Inspect the Lowest-Ranked Retrieved Results

One of the most important questions is:

> What happens when we increase `k`?

For each question, inspect the last result returned at `k=10`.

These results represent the additional context introduced by the larger retrieval window.

They may be useful, redundant or unrelated.

In [13]:
k10_results = similarity_results_df[
    similarity_results_df["K"] == 10
].sort_values(
    ["Question_ID", "Rank"]
)

for question_id in evaluation_questions:

    question_results = k10_results[
        k10_results["Question_ID"] == question_id
    ]

    last_result = question_results.iloc[-1]

    print("=" * 80)
    print(f"Question: {question_id}")
    print(f"Rank: {last_result['Rank']}")
    print(f"Row: {last_result['Source_Row']}")
    print(f"Score: {last_result['Score']:.4f}")
    print("Preview:")
    print(last_result["Content_Preview"])

Question: Q001
Rank: 10
Row: 283
Score: 0.6906
Preview:
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer ques
Question: Q002
Rank: 10
Row: 287
Score: 0.4472
Preview:
included prescription trends, customer behavior, business opportunities, formulary discussions, regional planning, call objectives, meeting outcomes, and action items. Product Overview: GARDASIL 9 is used in HPV Prevention. Clinical discussion covere
Question: Q003
Rank: 10
Row: 283
Score: 0.3811
Preview:
eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, com
Question: Q004
Rank: 10
Row: 284
Score: 0.3400
Preview:
eligibility, efficacy, 

## 14. Inspect One Complete `k=10` Result Set

Q004 was the weakest question for `text-embedding-3-large` in Notebook 03.

It is therefore useful to inspect how retrieval behaves when we expand the result set from 1 to 10.

This gives us a practical example of the retrieval trade-off.

In [14]:
q004_k10 = similarity_results_df[
    (
        similarity_results_df["Question_ID"] == "Q004"
    )
    & (
        similarity_results_df["K"] == 10
    )
].sort_values("Rank")

for _, row in q004_k10.iterrows():

    print(
        f"Rank {row['Rank']:>2} | "
        f"Row {row['Source_Row']} | "
        f"Score {row['Score']:.4f}"
    )

    print(row["Content_Preview"])
    print("-" * 80)

Rank  1 | Row 26 | Score 0.3400
eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, com
--------------------------------------------------------------------------------
Rank  2 | Row 71 | Score 0.3400
eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, com
--------------------------------------------------------------------------------
Rank  3 | Row 105 | Score 0.3400
eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature s

## 15. Retrieval Overlap Analysis

We can also check how many source rows are shared between different `k` values.

For example:

```text
k=1 ⊂ k=3 ⊂ k=5 ⊂ k=10
```

For a standard ranked similarity search, increasing `k` should preserve the earlier top-ranked results and add lower-ranked results.

This check validates the behavior of our retrieval configuration.

In [15]:
def get_source_rows(question_id, k):
    return set(
        similarity_results_df[
            (
                similarity_results_df["Question_ID"] == question_id
            )
            & (
                similarity_results_df["K"] == k
            )
        ]["Source_Row"]
    )

overlap_rows = []

for question_id in evaluation_questions:

    rows_k1 = get_source_rows(question_id, 1)
    rows_k3 = get_source_rows(question_id, 3)
    rows_k5 = get_source_rows(question_id, 5)
    rows_k10 = get_source_rows(question_id, 10)

    overlap_rows.append({
        "Question_ID": question_id,
        "K1_in_K3": rows_k1.issubset(rows_k3),
        "K3_in_K5": rows_k3.issubset(rows_k5),
        "K5_in_K10": rows_k5.issubset(rows_k10),
    })

overlap_df = pd.DataFrame(overlap_rows)

display(overlap_df)

,Question_ID,K1_in_K3,K3_in_K5,K5_in_K10
0,Q001,True,True,True
1,Q002,True,True,True
2,Q003,True,True,True
3,Q004,True,True,True


## 16. Findings

### What We Are Looking For

The experiment helps answer:

- Is `k=1` enough?
- Does `k=3` provide additional useful context?
- Does `k=5` add useful coverage?
- Does `k=10` introduce lower-relevance results?
- Is there a point where additional retrieval provides diminishing value?

### Important Principle

A higher `k` is not automatically better.

For a RAG application, the goal is:

```text
Enough relevant context
        +
Minimum unnecessary context
        ↓
Better downstream answer quality
```

The ideal `k` depends on the dataset and the downstream generation step.

## 17. Initial Retrieval Decision

Use the experiment results to select an initial retrieval baseline.

### Decision Criteria

Prefer the smallest `k` that:

1. Provides sufficient relevant coverage.
2. Does not introduce substantial low-score/noisy results.
3. Gives the downstream LLM enough context to answer the questions.

### Decision Template

After reviewing the generated results, record:

```text
Selected K:
3

Reason:
k=3 provides additional retrieval context compared with k=1 while maintaining virtually the same average relevance score.

Average Retrieved Score:
0.464805

k=1 Baseline:
0.464864

Difference vs k=1:
-0.0128%

Trade-off:
Increasing k from 1 to 10 produced only a very small reduction in average relevance score (-0.0281%). However, larger k values introduce additional retrieved chunks and therefore potentially increase downstream context size and processing cost.

Decision:
Use k=3 as the initial retrieval baseline because it provides a small amount of additional context without a meaningful reduction in retrieval relevance.
```

## 18. Persist Similarity Search Results

Detailed results are stored under:

```text
experiments/results/
```

Files:

- `similarity_search_results.csv`
- `similarity_search_summary.csv`
- `similarity_search_overlap.csv`

In [16]:
RESULTS_DIR = Path("../experiments/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results_path = RESULTS_DIR / "similarity_search_results.csv"
summary_path = RESULTS_DIR / "similarity_search_summary.csv"
overlap_path = RESULTS_DIR / "similarity_search_overlap.csv"

similarity_results_df.to_csv(
    results_path,
    index=False
)

k_summary.to_csv(
    summary_path,
    index=False
)

overlap_df.to_csv(
    overlap_path,
    index=False
)

print(f"Detailed results saved to: {results_path}")
print(f"Summary saved to: {summary_path}")
print(f"Overlap analysis saved to: {overlap_path}")

Detailed results saved to: ..\experiments\results\similarity_search_results.csv
Summary saved to: ..\experiments\results\similarity_search_summary.csv
Overlap analysis saved to: ..\experiments\results\similarity_search_overlap.csv


## 19. Update the Central Experiment Log

The `SS-*` experiments are added to the project's central experiment log.

Existing similarity-search records are replaced when the notebook is rerun.

In [17]:
EXPERIMENT_LOG_PATH = "../experiments/experiment_log.csv"

similarity_log = k_summary.copy()

similarity_log["Experiment_ID"] = (
    similarity_log["Experiment_ID"]
)

similarity_log["Area"] = "Similarity Search"

similarity_log["Configuration"] = (
    "text-embedding-3-large; "
    + similarity_log["K"].astype(str)
)

similarity_log["Question"] = "Q001-Q004"
similarity_log["Documents"] = len(data)
similarity_log["Chunks"] = len(chunks)
similarity_log["Top_K"] = similarity_log["K"]
similarity_log["Results"] = (
    similarity_log["Average_Retrieved_Score"]
    .round(4)
    .astype(str)
)

similarity_log["Observation"] = (
    "Compared retrieval depth across four multi-criteria questions."
)

similarity_log["Conclusion"] = (
    "Use the smallest k that provides sufficient relevant coverage "
    "without introducing excessive lower-ranked context."
)

log_columns = [
    "Experiment_ID",
    "Area",
    "Configuration",
    "Question",
    "Documents",
    "Chunks",
    "Top_K",
    "Results",
    "Observation",
    "Conclusion",
]

similarity_log = similarity_log[log_columns]

if (
    os.path.exists(EXPERIMENT_LOG_PATH)
    and os.path.getsize(EXPERIMENT_LOG_PATH) > 0
):

    existing_log = pd.read_csv(EXPERIMENT_LOG_PATH)

    if "Experiment_ID" in existing_log.columns:

        existing_log = existing_log[
            ~existing_log["Experiment_ID"].isin(
                ["SS-001", "SS-002", "SS-003", "SS-004"]
            )
        ]

        combined_log = pd.concat(
            [existing_log, similarity_log],
            ignore_index=True
        )

    else:
        combined_log = similarity_log

else:
    combined_log = similarity_log

combined_log.to_csv(
    EXPERIMENT_LOG_PATH,
    index=False
)

print(f"Experiment log updated: {EXPERIMENT_LOG_PATH}")
print(f"Total logged experiments: {len(combined_log)}")

Experiment log updated: ../experiments/experiment_log.csv
Total logged experiments: 9


## 20. Final Validation

The notebook is complete when:

- 300 documents are loaded.
- 1,538 chunks are generated.
- `text-embedding-3-large` is used consistently.
- Four questions are tested.
- `k=1`, `k=3`, `k=5` and `k=10` are evaluated.
- Every question returns the requested number of results.
- Results are persisted.

In [18]:
assert len(data) == 300
assert len(chunks) == 1538

assert len(evaluation_questions) == 4

assert set(
    similarity_results_df["K"].unique()
) == {1, 3, 5, 10}

for k in [1, 3, 5, 10]:
    expected_rows = len(evaluation_questions) * k

    actual_rows = len(
        similarity_results_df[
            similarity_results_df["K"] == k
        ]
    )

    assert actual_rows == expected_rows, (
        f"k={k}: expected {expected_rows} rows, "
        f"found {actual_rows}."
    )

assert not similarity_results_df.empty

print("✓ Similarity search experiment validation passed.")
print("✓ Embedding model: text-embedding-3-large")
print("✓ Questions evaluated: 4")
print("✓ k values evaluated: 1, 3, 5, 10")
print("✓ Results persisted.")
print("✓ Ready for Notebook 05 — RAG with LCEL.")

✓ Similarity search experiment validation passed.
✓ Embedding model: text-embedding-3-large
✓ Questions evaluated: 4
✓ k values evaluated: 1, 3, 5, 10
✓ Results persisted.
✓ Ready for Notebook 05 — RAG with LCEL.


## 21. Handoff to Notebook 05

The retrieval experiment is complete.

```text
300 Documents
      ↓
500 / 50 Chunking
      ↓
1,538 Chunks
      ↓
text-embedding-3-large
      ↓
4 Tricky Questions
      ↓
Similarity Search
      ↓
k = 1 / 3 / 5 / 10
      ↓
Retrieval Configuration
      ↓
Notebook 05
RAG with LCEL
```

Notebook 05 will use the selected retrieval configuration to build the complete:

```text
Question
   ↓
Retriever
   ↓
Context
   ↓
Prompt
   ↓
LLM
   ↓
Answer
```

RAG pipeline using the LangChain Expression Language concepts already covered in the course.